# S11 – BI/ML distribuido con Spark (evidencia real ESP32 + HC-SR04)
**Estudiante:** Lanzeloth Vargas Marichi

Este notebook **no lee Kafka**: calcula KPIs sobre la evidencia ya persistida por S9 (Parquet de observabilidad) y S10 (predicciones en tiempo real), los publica en un Pushgateway y los deja listos para Grafana.

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
import os

spark = (SparkSession.builder.appName("sesion11-bi-ml-distribuido")
         .master("local[*]").config("spark.ui.port", "4040").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RUTA_EVENTOS_S9 = "../s09-observabilidad/artifacts/atmos_real_observabilidad"
RUTA_PREDICCIONES_S10 = "../s10-series-tiempo/artifacts/s10_predicciones_real"
for r in (RUTA_EVENTOS_S9, RUTA_PREDICCIONES_S10):
    print(r, "->", "OK" if os.path.exists(r) else "NO EXISTE (revisa la ruta)")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/28 14:03:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


../s09-observabilidad/artifacts/atmos_real_observabilidad -> OK
../s10-series-tiempo/artifacts/s10_predicciones_real -> OK


## 1. Carga de la evidencia persistida (S9 y S10)

In [2]:
eventos = spark.read.parquet(RUTA_EVENTOS_S9)
predicciones = spark.read.parquet(RUTA_PREDICCIONES_S10)
eventos.printSchema()   # verifica que existan isValid y latencyMs; si se llaman distinto, ajusta la seccion 2
print(f"Filas eventos (S9): {eventos.count():,}")
print(f"Filas predicciones (S10): {predicciones.count():,}")
predicciones.orderBy("id").show(5, truncate=False)

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- distanciaCm: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = false)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)



Filas eventos (S9): 380


Filas predicciones (S10): 113


+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|1      |1836|2026-10-22 15:58:53|17.9 |41.05761474807243  |
|2      |1837|2026-10-22 15:58:56|17.9 |41.05761474807243  |
|2      |1838|2026-10-22 15:58:59|17.9 |41.05761474807243  |
|3      |1839|2026-10-22 15:59:02|17.9 |41.05761474807243  |
|4      |1840|2026-10-22 15:59:05|17.9 |41.05761474807243  |
+-------+----+-------------------+-----+-------------------+
only showing top 5 rows


## 2. KPI del flujo de eventos (capa BI)
Responde: *¿es bueno el dato que le llega al negocio?* Volumen, ratio de eventos válidos y latencia (promedio y p95).

In [3]:
kpi_eventos = eventos.agg(
    F.count("*").alias("total"),
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("avgLatencyMs"),
    F.expr("percentile_approx(latencyMs, 0.95)").alias("p95LatencyMs"),
).first()

print(f"Eventos procesados:   {kpi_eventos['total']}")
print(f"Ratio de validos:     {kpi_eventos['ratioValidos']:.4f}")
print(f"Latencia promedio:    {kpi_eventos['avgLatencyMs']:.2f} ms")
print(f"Latencia p95:         {kpi_eventos['p95LatencyMs']:.0f} ms")

[Stage 9:==========================================>                (5 + 2) / 7]

Eventos procesados:   380
Ratio de validos:     1.0000
Latencia promedio:    3350.63 ms
Latencia p95:         5838 ms


## 3. Error de predicción en producción (capa ML)
La predicción hecha en el evento *n* (para *n+1*) se compara con el valor real que llegó en el evento *n+1*. No hace falta ningún dato nuevo: el futuro de cada predicción ya está guardado en la propia tabla.
Se compara también contra la **persistencia** (predecir el valor actual) y contra el error de prueba offline de S10 (Regla #37 de Google: medir el *skew* entre entrenamiento y servicio).

In [4]:
w = Window.orderBy("id")
comparado = (predicciones
    .withColumn("real_siguiente", F.lead("valor", 1).over(w))
    .na.drop(subset=["real_siguiente"])
    .withColumn("errorAbsoluto", F.abs(F.col("prediccionSiguiente") - F.col("real_siguiente")))
    .withColumn("errorPersistencia", F.abs(F.col("valor") - F.col("real_siguiente"))))

comparado.orderBy("id").select("id", "fechaHora", "valor", "prediccionSiguiente", "real_siguiente", "errorAbsoluto").show(15, truncate=False)

kpi_prediccion = comparado.agg(
    F.count("*").alias("totalComparado"),
    F.avg("errorAbsoluto").alias("maeProduccion"),
    F.avg("errorPersistencia").alias("maePersistencia"),
).first()
ultima = predicciones.orderBy(F.desc("id")).first()

# Error de PRUEBA offline medido en S10 (conjunto de test cronologico)
# MAE de TEST medido en S10 (tabla de comparacion, conjunto de test cronologico)
MAE_TEST_LR = 0.1230
MAE_TEST_PERSISTENCIA = 0.1139

print(f"Predicciones comparables:     {kpi_prediccion['totalComparado']}")
print(f"MAE produccion (LR):          {kpi_prediccion['maeProduccion']:.4f}")
print(f"MAE produccion (persistencia):{kpi_prediccion['maePersistencia']:.4f}")
if MAE_TEST_LR:
    print(f"MAE prueba offline S10 (LR):  {MAE_TEST_LR}")
    print(f"Skew produccion/prueba (LR):  {kpi_prediccion['maeProduccion']/MAE_TEST_LR:.1f}x")
    print(f"MAE prueba offline S10 (persistencia): {MAE_TEST_PERSISTENCIA}")

+----+-------------------+-----+-------------------+--------------+------------------+
|id  |fechaHora          |valor|prediccionSiguiente|real_siguiente|errorAbsoluto     |
+----+-------------------+-----+-------------------+--------------+------------------+
|1836|2026-10-22 15:58:53|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1837|2026-10-22 15:58:56|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1838|2026-10-22 15:58:59|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1839|2026-10-22 15:59:02|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1840|2026-10-22 15:59:05|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1841|2026-10-22 15:59:08|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1842|2026-10-22 15:59:11|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1843|2026-10-22 15:59:14|17.9 |41.05761474807243  |17.9          |23.157614748072433|
|1844|2026-10-22 15:59:17|17.9 |41.05761474

## 4. Publicar las métricas en el Pushgateway
El notebook corre una vez y termina, por eso Prometheus no puede hacerle *scrape*: se empuja al Pushgateway con **PUT** (reemplaza todo el grupo `job/instance`) y Prometheus visita al Pushgateway.

In [5]:
import urllib.request

metricas = {
    "lambda26_bi_eventos_total": kpi_eventos["total"],
    "lambda26_bi_eventos_validos_ratio": kpi_eventos["ratioValidos"],
    "lambda26_bi_latencia_promedio_ms": kpi_eventos["avgLatencyMs"],
    "lambda26_bi_latencia_p95_ms": kpi_eventos["p95LatencyMs"],
    "lambda26_bi_prediccion_mae": kpi_prediccion["maeProduccion"],
    "lambda26_bi_prediccion_mae_persistencia": kpi_prediccion["maePersistencia"],
    "lambda26_bi_prediccion_ultima_real": ultima["valor"],
    "lambda26_bi_prediccion_ultima_predicha": ultima["prediccionSiguiente"],
}
lineas = []
for nombre, valor in metricas.items():
    lineas.append(f"# TYPE {nombre} gauge")
    lineas.append(f"{nombre} {float(valor)}")
cuerpo = "\n".join(lineas) + "\n"

PUSHGATEWAY_URL = "http://pushgateway:9091/metrics/job/lambda26_bi/instance/s11_notebook"
peticion = urllib.request.Request(PUSHGATEWAY_URL, data=cuerpo.encode("utf-8"), method="PUT")
with urllib.request.urlopen(peticion) as respuesta:
    print("Pushgateway respondio:", respuesta.status)

Pushgateway respondio: 200


## 5. Verificar en Prometheus con PromQL

In [6]:
import json, time
time.sleep(20)   # esperar al menos un scrape_interval (15 s)
for nombre in metricas:
    url = f"http://prometheus:9090/api/v1/query?query={nombre}"
    resultado = json.load(urllib.request.urlopen(url))
    valores = resultado["data"]["result"]
    print(f"{nombre}: {valores[0]['value'][1]}" if valores else f"{nombre}: SIN DATOS (revisa que pushgateway aparezca UP en Prometheus)")

lambda26_bi_eventos_total: 380
lambda26_bi_eventos_validos_ratio: 1
lambda26_bi_latencia_promedio_ms: 3350.6263157894737
lambda26_bi_latencia_p95_ms: 5838
lambda26_bi_prediccion_mae: 19.967977051082045
lambda26_bi_prediccion_mae_persistencia: 1.2535714285714283
lambda26_bi_prediccion_ultima_real: 107.5
lambda26_bi_prediccion_ultima_predicha: 110.01765608865338


## 6. Tablero de Grafana (manual, http://localhost:43000, admin/admin)
Dashboard nuevo **S11 - BI/ML distribuido** con estos paneles:

| Panel | Consulta | Visualización |
|---|---|---|
| Eventos procesados | `lambda26_bi_eventos_total` | Stat |
| Ratio de eventos válidos | `lambda26_bi_eventos_validos_ratio` | Stat (Unit: Percent 0.0-1.0) |
| Latencia promedio y p95 (ms) | `lambda26_bi_latencia_promedio_ms` y `lambda26_bi_latencia_p95_ms` | Stat o Time series |
| MAE en producción: LR vs persistencia | `lambda26_bi_prediccion_mae` y `lambda26_bi_prediccion_mae_persistencia` | Stat / Bar gauge |
| Última lectura real vs. predicha | `lambda26_bi_prediccion_ultima_real` y `lambda26_bi_prediccion_ultima_predicha` | Time series |

Son valores puntuales (el Pushgateway solo guarda el último PUT), por eso los paneles muestran líneas planas. Es lo esperado.